# gputherm on a free cloud GPU

Builds the CPU and CUDA solvers, verifies the GPU against the semi-analytic bioheat solution, then benchmarks CPU vs GPU.

**Colab:** Runtime → Change runtime type → T4 GPU. **Kaggle:** Settings → Accelerator → GPU.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
import os
if not os.path.exists('wearable-thermal-cfd'):
    !git clone --depth 1 https://github.com/aminilia/wearable-thermal-cfd
%cd wearable-thermal-cfd/gputherm
!pip -q install -r ../requirements.txt

## Build
`ARCH` is read from the GPU (T4 → `sm_75`, A100 → `sm_80`, L4 → `sm_89`).

In [ ]:
cap = !nvidia-smi --query-gpu=compute_cap --format=csv,noheader
arch = 'sm_' + cap[0].strip().replace('.', '')
print(arch)
!make all ARCH={arch}

## 1. Verify
Max cell-centre error vs the semi-analytic 1D bioheat solution on 3 grids; expect ~0.01 mK on the finest grid and observed order 2.00, identical to the CPU.

In [ ]:
!python python/verify_column.py --backend cpu
!python python/verify_column.py --backend cuda

## 2. Benchmark
Time per CG iteration and the effective memory bandwidth it implies (T4 peak ≈ 320 GB/s). Larger N takes longer on the CPU side.

In [ ]:
!OMP_NUM_THREADS=$(nproc) python python/bench.py --sizes 32 64 96 128 160

In [ ]:
from IPython.display import Image
Image('bench_results.png')

## 3. Profile
Nsight Systems shows the timeline (launch gaps, host round-trips for each dot product). Nsight Compute gives per-kernel memory throughput; free tiers sometimes block its hardware counters (`ERR_NVGPUCTRPERM`). If so, run this cell on a rented GPU instead.

In [ ]:
!nsys profile -o v0_timeline --force-overwrite true --stats=true ./bin/gputherm_cuda --bench 128 2>&1 | tail -40

In [ ]:
!ncu --kernel-name regex:k_apply --launch-count 1 --section SpeedOfLight --section MemoryWorkloadAnalysis ./bin/gputherm_cuda --bench 128 2>&1 | tail -40